In [2]:
from pathlib import Path

import strax
import abstrax
abstrax.print_versions()


import numba
import numpy as np

Initialize the same context as in the first two tutorials.

In [3]:
LONG_EMBER_RUN = '000001'

# Read-only example data shipped with abstrax (path relative to the tutorials folder)
TEST_DATA = Path('../test_data').resolve()
# Writable folder for everything we produce. Safe to delete.
OUTPUT_FOLDER = Path('./strax_data').resolve()

st = abstrax.contexts.abstrax_context(output_folder=OUTPUT_FOLDER)
st.storage.append(strax.DataDirectory(str(TEST_DATA), readonly=True))

# Strax Plugin, "An information transformer machine":

In tutorial 1, you have already seen what plugins are and how they interact with each other. In this notebook, we want to go a bit deeper into the technicalities and focus on the question: "How can I develop my own plugins?". 

However, before we go into detail, let's recap the most important facts you need to know about the data structure in strax:

* The data of a strax-based framework is organized in chunks and arrays. Each chunk represents a disjoint time interval containing the data as a [numpy structured array](https://numpy.org/doc/stable/user/basics.rec.html).
* The data (numpy structured array) can be imagined as a table in which each row represents a time interval. These time intervals are represented either by a `time` and `endtime` field (column) or a `time`, `dt` and `length` field. 
* The data within a chunk is sorted first by time and then by channel (if channel is provided). In your plugin, you must ensure that this order is kept in the end. 
* Zero-duration rows are not allowed as they may break some of the sorting algorithms.

Organizing data in such a scheme has the advantage that data can be processed in parallel loops as the information in a chunk is mostly independent of another chunk (This might not be true if you process a data type of short time intervals like peaks to larger time intervals like events. There is a dedicated plugin type for this.). 

In the subsequent sketch, we show as an example the symbolic data processing from the plugin "PulseProcessing" to "PeakProcessing". Each of the red and blue boxes represents a single chunk. The assembly belts represent the different parallel processes delivering the data. In the bottom right, the numpy structured array inside a chunk is shown. Since numpy can only handle fixed-length objects, things like the raw pulses coming from our digitizers have to be split across multiple rows.  


<center><img src="images/StraxChunks.png" width="1200"></center>

## What are the key ingredients of a Plugin?

### A Plugin Should Provide:
- A data type that it provides
- A way of computing the data type
- Necessary configuration options to set up the computation of the data type
- Additional functions to speed up the processing (although not strictly part of the plugin itself)

### Plugin types:

* strax.Plugin: This is the default plugin type. It can depend on one or more data_kinds.
* strax.OverlapWindowPlugin: An OverlapWindow plugin allows you to look for data in adjacent chunks. This only works for disjoint intervals like peaks. 
* strax.CutPlugin: A plugin which returns a boolean array that can be used to select or cut data. The output of a CutPlugin, if registered to the context, can be used in `get_array(run_id, target, selection='cut_plugin_provide')`. We will see more about this in tutorial 4.

For more information and additional plugin types, please refer to the [strax documentation](https://strax.readthedocs.io/en/latest/advanced/plugin_dev.html).

## Example Plugin:

Below you can find the full code of a typical default plugin.

It usually consists of:
- A docstring to say what the plugin does
- `depends_on`: the data types that the plugin takes as input
- `provides`: the output of the plugin, used in the `targets` argument in `get_array`
- Either `dtype` or an `infer_dtype()` method that provides the dtype for the structured array output
- A `compute(self, data_kind1, data_kind2, ...)` method that does the computation chunk by chunk.
- Functions that are called in `compute` to optimize the code (often using `numba`)

As example we use the `PeakBasics` plugin that provides `peak_basics`. In the following, we will walk through the structure of the plugin step by step.

```python
class PeakBasics(strax.Plugin):
    """Compute scalar summary properties for each peak.

    All structured-array fields (waveform data, per-channel areas, width
    arrays) are reduced to scalars so the output is pandas-friendly.
    """

    __version__ = "0.0.1"

    depends_on = "peaks"
    provides = "peak_basics"
    data_kind = "peaks"
    parallel = True
    compressor = "zstd"

    n_top_pmts = strax.URLConfig(
        default="detector://n_top_pmts",
        type=int,
        track=True,
        help="Number of top-array PMT channels (channels 0 … n_top_pmts-1).",
    )

    check_peak_sum_area_rtol = strax.URLConfig(
        default=None,
        track=False,
        infer_type=False,
        help=(
            "If set, assert that area_per_channel.sum() matches area within "
            "this relative tolerance (e.g. '1e-4').  None disables the check."
        ),
    )

    def infer_dtype(self):
        return strax.time_fields + [
            (("Weighted average center time of the peak [ns]", "center_time"), np.int64),
            (("Peak integral in PE", "area"), np.float32),
            (("Number of hits contributing at least one sample to the peak", "n_hits"), np.int32),
            (("Number of PMTs contributing to the peak", "n_channels"), np.int16),
            (("Number of top PMTs contributing to the peak", "top_n_channels"), np.int16),
            (("PMT number which contributes the most PE", "max_pmt"), np.int16),
            (("Area of signal in the largest-contributing PMT (PE)", "max_pmt_area"), np.float32),
            (("Total number of saturated channels", "n_saturated_channels"), np.int16),
            (("Width (in ns) of the central 50% area of the peak", "range_50p_area"), np.float32),
            (("Width (in ns) of the central 90% area of the peak", "range_90p_area"), np.float32),
            (("Fraction of area seen by the top array", "area_fraction_top"), np.float32),
            (("Length of the peak waveform in samples", "length"), np.int32),
            (("Time resolution of the peak waveform in ns", "dt"), np.int16),
            (("Weighted relative median time of the peak [ns]", "median_time"), np.float32),
            (("Time between 10% and 50% area quantiles [ns]", "rise_time"), np.float32),
            (("Number of PMTs with hits within tight range of mean", "tight_coincidence"), np.int16),
            (("Classification of the peak(let)", "type"), np.int8),
            (("Largest time difference between apexes of hits inside peak [ns]", "max_diff"), np.int32),
            (("Smallest time difference between apexes of hits inside peak [ns]", "min_diff"), np.int32),
            (("First channel/PMT number inside peak (sorted by apexes of hits)", "first_channel"), np.int16),
            (("Last channel/PMT number inside peak (sorted by apexes of hits)", "last_channel"), np.int16),
        ]

    def compute(self, peaks):
        r = np.zeros(len(peaks), self.dtype)
        if not len(peaks):
            return r

        p = peaks

        # Direct copies from peaks structured array.
        for field in (
            "time",
            "center_time",
            "length",
            "dt",
            "median_time",
            "area",
            "area_fraction_top",
            "type",
            "max_diff",
            "min_diff",
            "first_channel",
            "last_channel",
        ):
            r[field] = p[field]

        r["endtime"] = p["time"] + p["dt"] * p["length"]
        r["n_hits"] = p["n_hits"]
        r["tight_coincidence"] = p["tight_coincidence"]
        r["n_saturated_channels"] = p["n_saturated_channels"]

        r["n_channels"] = (p["area_per_channel"] > 0).sum(axis=1)
        r["top_n_channels"] = (p["area_per_channel"][:, : self.n_top_pmts] > 0).sum(axis=1)

        r["max_pmt"] = np.argmax(p["area_per_channel"], axis=1)
        r["max_pmt_area"] = np.max(p["area_per_channel"], axis=1)

        r["range_50p_area"] = p["width"][:, 5]
        r["range_90p_area"] = p["width"][:, 9]

        r["rise_time"] = -p["area_decile_from_midpoint"][:, 1]

        if self.check_peak_sum_area_rtol is not None:
            self.check_area(p["area_per_channel"].sum(axis=1), p, self.check_peak_sum_area_rtol)

        return r
```

In the following lines, we will go through the different aspects of a plugin and explain them one by one.

## Plugin Type, Header, and dtype

The first thing we usually define in a plugin is some meta-level information, which strax requires to sort the plugin into the processing stream at the right position. Furthermore, this information is used to define the lineage of the plugin.

```python
class PeakBasics(strax.Plugin):
    """Compute scalar summary properties for each peak.

    All structured-array fields (waveform data, per-channel areas, width
    arrays) are reduced to scalars so the output is pandas-friendly.
    """

    __version__ = "0.0.1"

    depends_on = "peaks"
    provides = "peak_basics"
    data_kind = "peaks"
    parallel = True
    compressor = "zstd"
```

The class itself inherits from one of the supported strax.Plugin types. In this example, it is the base strax.Plugin class. Each Plugin requires either a tuple or a string of data_types the plugin `depends_on`. The same holds true for its return which is indicated by `provides`. By default, the new `data_type` returned by a plugin has the same data_kind as the plugin input. If this is not desired, one has to change the `data_kind` into a new data_kind. If a plugin depends on or provides more than one data type, you have to specify them as a tuple. This information is used by strax to automatically insert your plugin at the right position of the processing stream.

In order to be able to track changes in the plugin code itself, each plugin has its own `__version__`. Whenever you make a change in the code that changes the result of the plugin, you should increment the version to indicate this. If you forget to change the version, the lineage of the plugin will stay the same and strax will silently return old data to you! During development, you can also set the version to None to avoid incrementing the version every time you make a little change.

Besides these mandatory fields, there are a bunch of optional settings like the `parallel=True` flag, which indicates whether the chunks of a plugin can be processed in parallel, or the `compressor` parameter which tells strax what kind of compressor to use when writing a strax chunk to disk. Depending on the type of compressor, data compression/decompression may take longer but save more disk space. To get an overview of all possible options, you can check the [strax code](https://github.com/AxFoundation/strax/blob/58544a7d829178019b78dbcd31d4bc1626eda382/strax/plugin.py#L41-L99) directly.  

## URL Configs

You already learned about the basic idea of URL configs in tutorial 1. Each plugin may come with a set of plugin-dependent settings that you do not want to hardcode in your processing chain. These parameters can be configured via URL configs and can range from simple scalar values to three-dimensional reconstruction maps. Our example plugin takes two URL configs. 

```python
    n_top_pmts = strax.URLConfig(
        default="detector://n_top_pmts",
        type=int,
        track=True,
        help="Number of top-array PMT channels (channels 0 … n_top_pmts-1).",
    )

    check_peak_sum_area_rtol = strax.URLConfig(
        default=None,
        track=False,
        infer_type=False,
        help=(
            "If set, assert that area_per_channel.sum() matches area within "
            "this relative tolerance (e.g. '1e-4').  None disables the check."
        ),
    )
```

The first config reads the detector config file `detector.yaml` in `abstrax/config/` and returns the value of `n_top_pmts`. This config value is of type integer and is tracked by strax. This means whenever we change `n_top_pmts` the lineage of the `peak_basics` data will change. Each URLConfig is also associated with a `help` statement that is listed when calling `st.show_config`. The second config is only used for debugging and testing purposes, so consequently its state is not tracked in the lineage.

If you want to learn more about how URL configs work and how you can check what they return, please refer to the first tutorial notebook.

## Plugin Initialization

During the initialization of a plugin, the `infer_dtype` and `setup` methods are executed first before processing the data. These methods are purely optional, and in fact one could specify the `dtype` directly.

The `infer_dtype` method is called first and helps to define the [dtype](https://numpy.org/doc/stable/user/basics.rec.html#structured-datatype-creation) of the structured array provided by the plugin. This is especially handy if the plugin returns multiple different data_kinds. 
```python
    def infer_dtype(self):
        return strax.time_fields + [
            (("Weighted average center time of the peak [ns]", "center_time"), np.int64),
            (("Peak integral in PE", "area"), np.float32),
            (("Number of hits contributing at least one sample to the peak", "n_hits"), np.int32),
            (("Number of PMTs contributing to the peak", "n_channels"), np.int16),
            (("Number of top PMTs contributing to the peak", "top_n_channels"), np.int16),
            (("PMT number which contributes the most PE", "max_pmt"), np.int16),
            (("Area of signal in the largest-contributing PMT (PE)", "max_pmt_area"), np.float32),
            (("Total number of saturated channels", "n_saturated_channels"), np.int16),
            (("Width (in ns) of the central 50% area of the peak", "range_50p_area"), np.float32),
            (("Width (in ns) of the central 90% area of the peak", "range_90p_area"), np.float32),
            (("Fraction of area seen by the top array", "area_fraction_top"), np.float32),
            (("Length of the peak waveform in samples", "length"), np.int32),
            (("Time resolution of the peak waveform in ns", "dt"), np.int16),
            (("Weighted relative median time of the peak [ns]", "median_time"), np.float32),
            (("Time between 10% and 50% area quantiles [ns]", "rise_time"), np.float32),
            (("Number of PMTs with hits within tight range of mean", "tight_coincidence"), np.int16),
            (("Classification of the peak(let)", "type"), np.int8),
            (("Largest time difference between apexes of hits inside peak [ns]", "max_diff"), np.int32),
            (("Smallest time difference between apexes of hits inside peak [ns]", "min_diff"), np.int32),
            (("First channel/PMT number inside peak (sorted by apexes of hits)", "first_channel"), np.int16),
            (("Last channel/PMT number inside peak (sorted by apexes of hits)", "last_channel"), np.int16),
        ]
```
The `infer_dtype` method always returns a list of the columns a data type should contain.


The setup method allows the user to define certain parameters which will always stay the same during the processing, or add some logic checks for some of the settings. Since `PeakBasics` does not have a setup method, here is an excerpt of the setup method of `Peaklets`, which checks a setting that may break the processing if it is configured wrongly.
```python    
    def setup(self):
        if self.peak_min_pmts > 2:
            # Raising this breaks the lone_hit definition; see straxen/issues/295.
            raise NotImplementedError(
                f"peak_min_pmts={self.peak_min_pmts} interferes with lone_hit definition."
            )
        (...)
```

## Compute & Optimized code:

Last but not least, we have the `compute` method and your optimized code which is used to process/transform the information from one data_type into another.

```python
    def compute(self, peaks):
        r = np.zeros(len(peaks), self.dtype)
        if not len(peaks):
            return r

        p = peaks

        # Direct copies from peaks structured array.
        for field in (
            "time",
            "center_time",
            "length",
            "dt",
            "median_time",
            "area",
            "area_fraction_top",
            "type",
            "max_diff",
            "min_diff",
            "first_channel",
            "last_channel",
        ):
            r[field] = p[field]

        r["endtime"] = p["time"] + p["dt"] * p["length"]
        r["n_hits"] = p["n_hits"]
        r["tight_coincidence"] = p["tight_coincidence"]
        r["n_saturated_channels"] = p["n_saturated_channels"]

        r["n_channels"] = (p["area_per_channel"] > 0).sum(axis=1)
        r["top_n_channels"] = (p["area_per_channel"][:, : self.n_top_pmts] > 0).sum(axis=1)

        r["max_pmt"] = np.argmax(p["area_per_channel"], axis=1)
        r["max_pmt_area"] = np.max(p["area_per_channel"], axis=1)

        r["range_50p_area"] = p["width"][:, 5]
        r["range_90p_area"] = p["width"][:, 9]

        r["rise_time"] = -p["area_decile_from_midpoint"][:, 1]

        if self.check_peak_sum_area_rtol is not None:
            self.check_area(p["area_per_channel"].sum(axis=1), p, self.check_peak_sum_area_rtol)

        return r
```

You can imagine the compute statement as a little script which is automatically applied to each chunk of the `data_type` the plugin depends on when making the new `provides` `data_type`. The return of the compute method is either a numpy structured array of the dtype you defined earlier or, in the case of a plugin with multiple outputs, a dictionary with the `provides` names as keys and arrays as the dictionary values, e.g. like in [PulseProcessing](https://github.com/XENONnT/straxen/blob/9a53370df918454ef7cec47be5ed685b97122568/straxen/plugins/records/records.py#L205-L207). 

Some comments about the compute statement:
* In addition to the `depends_on` data_kind fields, the compute method can also contain `start` and `end` as function arguments. `start` and `end` represent the `start` and `end` time of the corresponding chunk. This may be helpful if you want to compute a rate or make sure that your new data_type does not extend beyond the chunk boundaries.
* The arguments of the `compute` are named and sorted according to the `data_kinds` the plugin depends on. E.g. if you write a plugin like `straxen.EventBasics` which `depends_on = ('events', 'peak_basics', 'peak_positions', 'peak_proximity')`, your compute method looks like `def compute(self, events, peaks):` If your plugin only `depends_on = ('peak_basics', 'peak_positions')` it will be just `peaks`, etc. To see the differences for a plugin with [multiple dependencies](https://github.com/XENONnT/straxen/blob/e0bf0f21c797ec0008bc13b779d6c00a6ae4fcbb/straxen/plugins/peaklet_processing.py#L510) or [multiple provides](https://github.com/XENONnT/straxen/blob/e0bf0f21c797ec0008bc13b779d6c00a6ae4fcbb/straxen/plugins/pulse_processing.py#L97-L142) you can check the linked plugins.
* For some of our plugins, we re-chunk our data after processing. This helps to reduce the overall number of files as the information quality gets more and more refined. In case your plugin depends on multiple differently chunked data_kinds, strax will automatically find suitable gaps for you: 

![Example image strax chunking](images/MultipleChunks.png)


In addition to your compute method, you can define optimized code for your processing step outside of the plugin, but within the same `.py` file if you like. Functions that could be of general interest can also become part of strax itself. We recommend optimizing your code as much as possible using `numba`, which allows you to compile Python and numpy-based code just-in-time to machine code. This will speed up the computation time significantly. Writing `numba`-compilable code follows its own rules, and we recommend familiarizing yourself with them using some other tutorial material. 

# How can I develop and test plugins?

Now that we have learned a lot about plugins in theory, let us focus a bit more on how to develop and handle new plugins. Typically, a plugin evolves from a study you did, for example, in a notebook. You can imagine the `compute` method in a plugin as a notebook cell in which you did something like:

```python
data = st.get_array(run_id, 'data_kind', seconds_range=(0, 20))

# def compute(self, data_kind):
data_after_a = functionA(data)
data_after_b = functionB(data_after_a)
# return data_after_b
```
But instead of computing `data_after_b` only for the first 20 s (maybe for memory reasons) you would like to compute it for the whole run or even an entire science run containing hundreds of runs. In this case, you should develop your own plugin. In the following, we will create a new simple example plugin.

Currently, abstrax only has a placeholder plugin for the position reconstruction returning the position of all peaks as (0, 0):

In [5]:
peak_positions = st.get_array(LONG_EMBER_RUN, 'peak_positions')

Loading peak_positions: |          | 0.00 % [00:00<?]

In [6]:
peak_positions['x']

array([0., 0., 0., ..., 0., 0., 0.], shape=(20410,), dtype=float32)

Let's replace this plugin with a simple area-weighted average of the PMT positions:

In [7]:
class SimplePositionReconstruction(strax.Plugin):
    """
    This plugin computes the position of an S2 peak as the
    weighted average over all PMT channels. The plugin uses 
    the convention that the first N PMTs are located in the 
    top PMT array.
    """
    __version__ = '0.0.1'
    
    depends_on = 'peaks'
    provides = 'peak_positions'

    n_top_pmts = strax.URLConfig(
        default="detector://n_top_pmts",
        type=int,
        track=True,
        help="Number of top-array PMT channels (channels 0 … n_top_pmts-1).",
    )

    pmt_channel_map = strax.URLConfig(
        default="pkg://pmt_positions.csv",
        track=True,
        help="Pandas DataFrame with PMT positions.",
    )

    parallel = True
    
    def infer_dtype(self):
        dtype = strax.time_fields + [
            (("Reconstructed X position of the peak, uncorrected [cm]", "x"), np.float32),
            (("Reconstructed Y position of the peak, uncorrected [cm]", "y"), np.float32),
        ]
        return dtype
    
    def compute(self, peaks):    
        
        result = np.zeros(
            len(peaks),
            self.dtype
        ) 
        result['time'] = peaks['time']
        result['endtime'] = strax.endtime(peaks)

        pmt_x_position = self.pmt_channel_map['x'].values[:self.n_top_pmts]
        pmt_y_position = self.pmt_channel_map['y'].values[:self.n_top_pmts]

        get_area_weighted_average(
            peaks, 
            result,
            pmt_x_position,
            pmt_y_position,
        )
        
        return result

@numba.njit
def get_area_weighted_average(
        peaks, 
        peak_positions,
        x_positions, 
        y_positions
        ):
    """Computes area weighted position of an S2 peak. Modifies
    peak positions array in place. 
    """ 
    n_top_pmts = len(x_positions)
    for index, peak in enumerate(peaks):
        
        _is_not_an_s2_peak = (peak['type'] != 2)
        if _is_not_an_s2_peak:
            continue

        area_per_channel = peak['area_per_channel'][:n_top_pmts]

        total_area = np.sum(area_per_channel)
        _is_not_seen_by_top_array = (total_area == 0)
        if _is_not_seen_by_top_array:
            continue

        x_mean = np.sum(x_positions*area_per_channel)/total_area
        y_mean = np.sum(y_positions*area_per_channel)/total_area
        peak_positions['x'][index] = x_mean
        peak_positions['y'][index] = y_mean


Before we register the new plugin, let's have a look at the lineage:

In [8]:
st.key_for(LONG_EMBER_RUN, 'peak_positions').lineage

{'peak_positions': ('PeakPositions', '0.0.1', {}),
 'peaks': ('Peaks', '0.0.1', {'merge_without_s1': True}),
 'lone_hits': ('Peaklets',
  '0.0.2',
  {'gain_model': 'numpy:///mnt/d/XLZD/mymodules/abstrax/abstrax/resources/fake_to_pe_494_0.005.npy',
   'n_tpc_pmts': 'detector://n_tpc_pmts',
   'n_top_pmts': 'detector://n_top_pmts',
   'hit_min_amplitude': 15,
   'peaklet_gap_threshold': 700,
   'peak_left_extension': 30,
   'peak_right_extension': 200,
   'peak_min_pmts': 2,
   'peaklet_max_duration': 10000000,
   'peak_split_gof_threshold': (None,
    ((0.5, 1.0), (6.0, 0.4)),
    ((2.5, 1.0), (5.625, 0.4))),
   'peak_split_filter_wing_width': 70,
   'peak_split_min_area': 40.0,
   'peak_split_iterations': 20,
   'saturation_correction_on': True,
   'saturation_reference_length': 100,
   'saturation_min_reference_length': 20,
   'tight_coincidence_window_left': 50,
   'tight_coincidence_window_right': 50,
   'store_data_top': True,
   'store_data_start': True}),
 'pulse_counts': ('Pulse

`peak_positions` is currently still computed by `PeakPositions`, so let us register the new plugin:

In [9]:
st.register(SimplePositionReconstruction)

__main__.SimplePositionReconstruction

Now that the new plugin is registered, we can check the lineage again:

In [10]:
st.key_for(LONG_EMBER_RUN, 'peak_positions').lineage

{'peak_positions': ('SimplePositionReconstruction',
  '0.0.1',
  {'n_top_pmts': 'detector://n_top_pmts',
   'pmt_channel_map': 'pkg://pmt_positions.csv'}),
 'peaks': ('Peaks', '0.0.1', {'merge_without_s1': True}),
 'lone_hits': ('Peaklets',
  '0.0.2',
  {'gain_model': 'numpy:///mnt/d/XLZD/mymodules/abstrax/abstrax/resources/fake_to_pe_494_0.005.npy',
   'n_tpc_pmts': 'detector://n_tpc_pmts',
   'n_top_pmts': 'detector://n_top_pmts',
   'hit_min_amplitude': 15,
   'peaklet_gap_threshold': 700,
   'peak_left_extension': 30,
   'peak_right_extension': 200,
   'peak_min_pmts': 2,
   'peaklet_max_duration': 10000000,
   'peak_split_gof_threshold': (None,
    ((0.5, 1.0), (6.0, 0.4)),
    ((2.5, 1.0), (5.625, 0.4))),
   'peak_split_filter_wing_width': 70,
   'peak_split_min_area': 40.0,
   'peak_split_iterations': 20,
   'saturation_correction_on': True,
   'saturation_reference_length': 100,
   'saturation_min_reference_length': 20,
   'tight_coincidence_window_left': 50,
   'tight_coincide

The plugin name in the lineage is updated so let's try the new plugin. However, before trying the new plugin on an entire run, it is usually recommended to test the code on a small data sample first. For this we initialize the plugin and load a small set of the data it depends on. Then we can simply call the compute method and check the result:

In [11]:
plugin = st.get_single_plugin(LONG_EMBER_RUN, 'peak_positions')

peaks = st.get_array(LONG_EMBER_RUN, 'peaks', seconds_range=(0, 1))
plugin.compute(peaks)

Loading peaks: |          | 0.00 % [00:00<?]

array([(       40,       500,   0.       ,   0.       ),
       (   287430,    293870,   9.237011 ,  38.15156  ),
       ( 20000030,  20000600,   0.       ,   0.       ),
       ( 20006400,  20009680,  -8.627974 ,   4.5707583),
       ( 40000020,  40000560,   0.       ,   0.       ),
       ( 41064090,  41073740,  18.051302 ,  42.07877  ),
       ( 60000030,  60000940,   0.       ,   0.       ),
       ( 60838280,  60847080,  38.80716  ,   9.256555 ),
       ( 80000040,  80000550,   0.       ,   0.       ),
       ( 80446660,  80454100, -28.223892 , -34.67291  ),
       (100000040, 100000530,   0.       ,   0.       ),
       (100648170, 100656050,  10.325012 ,  44.90793  ),
       (120000030, 120000520,   0.       ,   0.       ),
       (120401880, 120408680,  34.719322 ,  19.575115 ),
       (140000040, 140001030,   0.       ,   0.       ),
       (140458200, 140466200,  -6.6960325,  45.7022   ),
       (160000030, 160000490,   0.       ,   0.       ),
       (160235420, 160241980,  

If we are happy with the result, we can run the plugin on an entire run:

In [12]:
peak_positions = st.get_array(LONG_EMBER_RUN, 'peak_positions')
peak_positions

Loading peak_positions: |          | 0.00 % [00:00<?]

array([(          40,          500,  0.      ,   0.      ),
       (      287430,       293870,  9.237011,  38.15156 ),
       (    20000030,     20000600,  0.      ,   0.      ), ...,
       (199960031430, 199960035050,  0.825482, -11.290235),
       (199980000040, 199980000770,  0.      ,   0.      ),
       (199980939940, 199980948940, 19.809063,  -8.86886 )],
      shape=(20410,), dtype=[(('Start time since unix epoch [ns]', 'time'), '<i8'), (('Exclusive end time since unix epoch [ns]', 'endtime'), '<i8'), (('Reconstructed X position of the peak, uncorrected [cm]', 'x'), '<f4'), (('Reconstructed Y position of the peak, uncorrected [cm]', 'y'), '<f4')])

Now you have learned how to develop your own plugin. As you saw, it is quite simple. One more word of advice on developing new plugins: if you notice any bugs during development and make any modifications to the plugin, it is advisable to restart your context to make sure that nothing residual of your failed processing attempt is left in the strax processor.